In [0]:
import dlt
from pyspark.sql.functions import col, lit, sum, count
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DoubleType

In [0]:
dbutils.fs.mkdirs("/Volumes/dbcertification/greatlearning/dltvolume/input/")

In [0]:
schema_csv = StructType([
    StructField("InvoiceNo", StringType(),True),
    StructField("StockCode",StringType(),True),
    StructField("Description",StringType(),True),
    StructField("Quantity",IntegerType(),True),
    StructField("InvoiceDate",TimestampType(),True),
    StructField("UnitPrice",DoubleType(),True),
    StructField("CustomerID",IntegerType(),True)
])

In [0]:
cloud_file_options = {
    "cloudFiles.format" : "CSV",
    "pathGlobFilter" : "*.csv",
    "header": "true"
}

In [0]:
@dlt.table
def bronze_load():
    df = (spark
        .readStream
        .format("cloudFiles")
        .options(**cloud_file_options)
        .schema(schema_csv)
        .load("/Volumes/dbcertification/greatlearning/dltvolume/input/")
        )
    return df

In [0]:
@dlt.table
def bronze_static_load():
    df = (spark
        .read
        .format("csv")
        .options(**cloud_file_options)
        .schema(schema_csv)
        .load("/Volumes/dbcertification/greatlearning/dltvolume/input/")
        )
    return df

In [0]:
# view from streaming table
@dlt.view
def dataquality_view():
    df = dlt.readStream("bronze_load") 
    return df

In [0]:
# view from another table
@dlt.view
def dataquality_view2():
    df = dlt.readStream ("bronze_static_load") 
    return df